In [1]:
!nvidia-smi

Mon Nov 23 11:18:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

transformers: 5.20.2
datasets: 4.8.5


In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("turkish-nlp-suite/MusteriYorumlari")

train = ds["train"].to_pandas()
val = ds["validation"].to_pandas()
test = ds["test"].to_pandas()

def to_sentiment(label):
    if label in (0, 1):      # 1-2 stars
        return "Negative"
    elif label == 2:         # 3 stars
        return "Neutral"
    else:                    # 4-5 stars
        return "Positive"

for df in (train, val, test):
    df["sentiment"] = df["label"].map(to_sentiment)

print(train.shape, val.shape, test.shape)
print(train["sentiment"].value_counts())

README.md:   0%|          | 0.00/2.85k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

valid.jsonl:   0%|          | 0.00/1.87M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.86M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/73920 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15000 [00:00<?, ? examples/s]

(73920, 3) (15000, 3) (15000, 3)
sentiment
Positive    43885
Negative    17152
Neutral     12883
Name: count, dtype: int64


In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = "dbmdz/bert-base-turkish-cased"
MAX_LENGTH = 128
LABELS = ["Negative", "Neutral", "Positive"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_dataset(df):
    encoding = tokenizer(
        df["text"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    labels = torch.tensor([label2id[s] for s in df["sentiment"]])
    return TensorDataset(encoding["input_ids"], encoding["attention_mask"], labels)

train_ds = make_dataset(train)
val_ds = make_dataset(val)
test_ds = make_dataset(test)

print(len(train_ds), len(val_ds), len(test_ds))
print("Example token ids:", train_ds[0][0][:12])
print("Decoded back:", tokenizer.decode(train_ds[0][0][:12]))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

73920 15000 15000
Example token ids: tensor([    2, 12376,  2146, 12622,  2133,  3065,  4743, 23233,  4663,  3658,
         3065, 18071])
Decoded back: [CLS] Ürünleri 2025 olarak göndereceğiz dedikleri halde öyle gönderilmemiş


In [5]:
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
).to(device)

# Class weights to counter the imbalance identified as RISK-03
counts = train["sentiment"].value_counts()
weights = torch.tensor(
    [len(train) / (3 * counts[label]) for label in LABELS],
    dtype=torch.float,
).to(device)
print("Class weights:", {l: round(w, 3) for l, w in zip(LABELS, weights.tolist())})

Device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: {'Negative': 1.437, 'Neutral': 1.913, 'Positive': 0.561}


In [7]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 2e-5

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, 0, total_steps)
loss_fn = CrossEntropyLoss(weight=weights)
scaler = torch.amp.GradScaler("cuda")

print(f"Steps per epoch: {len(train_loader)} | Total steps: {total_steps}")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for step, (input_ids, attention_mask, labels) in enumerate(progress, start=1):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        progress.set_postfix(loss=f"{running_loss / step:.4f}")

    print(f"Epoch {epoch + 1} average training loss: {running_loss / len(train_loader):.4f}")

Steps per epoch: 2310 | Total steps: 4620


Epoch 1/2:   0%|          | 0/2310 [00:00<?, ?it/s]

Epoch 1 average training loss: 0.5671


Epoch 2/2:   0%|          | 0/2310 [00:00<?, ?it/s]

Epoch 2 average training loss: 0.4485


In [8]:
from sklearn.metrics import classification_report, f1_score
import numpy as np

@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in tqdm(loader, desc="Evaluating"):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = evaluate(val_loader)

print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))
print("Macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 3))

Evaluating:   0%|          | 0/235 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    Negative      0.853     0.822     0.838      3616
     Neutral      0.545     0.747     0.630      2629
    Positive      0.946     0.854     0.898      8755

    accuracy                          0.828     15000
   macro avg      0.781     0.808     0.788     15000
weighted avg      0.853     0.828     0.836     15000

Macro-F1: 0.788


In [9]:
y_true_test, y_pred_test = evaluate(test_loader)
print(classification_report(y_true_test, y_pred_test, target_names=LABELS, digits=3))
print("Test Macro-F1:", round(f1_score(y_true_test, y_pred_test, average="macro"), 3))

Evaluating:   0%|          | 0/235 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    Negative      0.844     0.797     0.820      3577
     Neutral      0.529     0.734     0.615      2542
    Positive      0.941     0.858     0.898      8881

    accuracy                          0.823     15000
   macro avg      0.772     0.796     0.778     15000
weighted avg      0.848     0.823     0.831     15000

Test Macro-F1: 0.778


In [10]:
import shutil

SAVE_DIR = "/kaggle/working/berturk-sentiment"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved files:")
!ls -lh {SAVE_DIR}

shutil.make_archive("/kaggle/working/berturk-sentiment", "zip", SAVE_DIR)
print("\nArchive:")
!ls -lh /kaggle/working/berturk-sentiment.zip

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved files:
total 423M
-rw-r--r-- 1 root root  848 Nov 23 12:00 config.json
-rw------- 1 root root 423M Nov 23 12:00 model.safetensors
-rw-r--r-- 1 root root  370 Nov 23 12:00 tokenizer_config.json
-rw-r--r-- 1 root root 738K Nov 23 12:00 tokenizer.json

Archive:
-rw-r--r-- 1 root root 392M Nov 23 12:00 /kaggle/working/berturk-sentiment.zip
